In [0]:
USE DATABASE completejourney

In [0]:
CREATE TABLE tdata AS
SELECT * FROM transaction_data WHERE store_id IN (367,406,381)

num_affected_rows,num_inserted_rows


In [0]:
SELECT store_id, MIN(day), MAX(day)
      , MAX(day) - MIN(day)
      , COUNT(DISTINCT day) 
FROM tdata
GROUP BY 1

store_id,MIN(day),MAX(day),MAX(day)-MIN(day),COUNT(DISTINCTday)
367,3,711,708,692
406,3,711,708,687
381,4,711,707,697


In [0]:
CREATE TABLE all_days AS
SELECT store_id, day
FROM (
  SELECT explode(sequence(4,711)) AS day
) JOIN (
  SELECT DISTINCT store_id FROM tdata
) 
ON TRUE
ORDER BY store_id, day

num_affected_rows,num_inserted_rows


In [0]:
SELECT store_id, day, sales
FROM all_days
LEFT JOIN (
  SELECT store_id, day, SUM(sales_value) sales
  FROM tdata
  GROUP BY 1,2
)
USING (store_id, day)
WHERE store_id = 381

store_id,day,sales
381,4,5.67
381,5,null
381,6,15.57
381,7,null
381,8,null
381,9,14.14
381,10,19.89
381,11,16.31
381,12,88.08999999999999
381,13,3.97


Databricks visualization. Run in Databricks to view.

In [0]:
CREATE TABLE features AS 
WITH sales_value_ts (
  SELECT day, store_id, COALESCE(SUM(sales_value),0) as sv
  FROM all_days
  JOIN tdata
  USING (day, store_id)
  GROUP BY day, store_id
)
SELECT day
      , store_id
      , sv AS to_predict
      , LAG(sv, 1) OVER (PARTITION BY store_id ORDER BY day) AS sv_1
      , LAG(sv, 7) OVER (PARTITION BY store_id ORDER BY day) AS sv_7
      , LAG(sv, 30) OVER (PARTITION BY store_id ORDER BY day) AS sv_30
      , SUM(sv) OVER (PARTITION BY store_id ORDER BY day ROWS BETWEEN 7 PRECEDING AND CURRENT ROW) - sv AS sv_w1
      , SUM(sv) OVER (PARTITION BY store_id ORDER BY day ROWS BETWEEN 30 PRECEDING AND CURRENT ROW) - sv AS sv_m1
      , day % 30 AS wom 
      , day % 364 AS woy
      , day % 7 AS dow
FROM sales_value_ts
WHERE day > 100
QUALIFY sv_1 is not null
AND sv_7 is not null
AND sv_30 is not null
AND sv_w1 is not null
AND sv_m1 is not null
ORDER BY store_id, day



num_affected_rows,num_inserted_rows


In [0]:
CREATE TABLE features_with_index AS
SELECT row_number() OVER (ORDER BY store_id, day) -1 AS row_index, *
FROM features
ORDER BY store_id, day

num_affected_rows,num_inserted_rows


In [0]:
%python
def get_temporal_X_y(days_to_fetch, debug = False):

    if not isinstance(days_to_fetch, list):
        days_to_fetch = [days_to_fetch]

    str_lst = ','.join( [ str(x) for x in days_to_fetch ] )

    sql = f"""
    SELECT row_index
    FROM features_with_index
    WHERE day IN ({str_lst})
    """

    if debug:
        print(sql)
    
    df = spark.sql(sql).toPandas()
    indexes = df.row_index.to_numpy()
    return indexes

get_temporal_X_y(711)


array([ 568, 1147, 1725], dtype=int32)

In [0]:
%python
list(range(0,10))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [0]:
%python
def get_temporal_valid_train_X_y(day_at, debug = False):
    
    valid_indices = get_temporal_X_y(day_at, debug = debug)
    train_indices = get_temporal_X_y( list(range(day_at)), debug = debug)



    return (train_indices, valid_indices)

test_indices = get_temporal_X_y(711)

validation_sets = []
for i in range(20):
    validation_sets.append(get_temporal_valid_train_X_y(710-(i*10)))




In [0]:
%python
sql = """
SELECT * EXCEPT (row_index)
FROM features_with_index
ORDER BY row_index
"""
data = spark.sql(sql).toPandas()
X = data.drop(columns = ['to_predict','day','store_id'])
y = data.to_predict.to_numpy()

In [0]:
%python
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error

rf = RandomForestRegressor()

cross_val_score(rf, X, y, cv = validation_sets, scoring = 'neg_mean_absolute_percentage_error') * -1

array([0.33847809, 2.5698043 , 0.55025183])

In [0]:
%python
from sklearn.model_selection import GridSearchCV

rf = RandomForestRegressor()

grid = {
    'max_depth': [4,10,100, 1000],
    'max_features': [1, 'log2', 'sqrt']
}

rfgs = GridSearchCV(rf, grid, cv =validation_sets ,scoring='neg_mean_absolute_percentage_error')

_ = rfgs.fit(X, y)


In [0]:
%python
np.mean(rfgs.cv_results_["mean_test_score"])

np.float64(-1.0000679220044004)

In [0]:
%python
rfgs.best_params_

{'max_depth': 4, 'max_features': 'log2'}

In [0]:
%python


20

In [0]:
%python
import numpy as np
r = []
for i in range(len(validation_sets)):
    r.append( mean_absolute_percentage_error(y[validation_sets[i][1]], X.iloc[validation_sets[i][1]].sv_1.to_numpy()) )
print(np.mean(r))

1.0942980289236452


In [0]:
%python
validation_sets

[(array([   0,    1,    2, ..., 1721, 1722, 1723], dtype=int32),
  array([ 567, 1146, 1724], dtype=int32)),
 (array([   0,    1,    2, ..., 1711, 1712, 1713], dtype=int32),
  array([ 557, 1136, 1714], dtype=int32)),
 (array([   0,    1,    2, ..., 1701, 1702, 1703], dtype=int32),
  array([ 547, 1126, 1704], dtype=int32))]

In [0]:
%python
last_day = 711
X_test, y_test = get_temporal_X_y(last_day)
X_valid, y_valid = get_temporal_X_y(last_day-2)
X_train, y_train = get_temporal_X_y(last_day-3)